In [ ]:
import argparse
import logging
import pickle as pkl
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from lightning import seed_everything
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset, Subset
from torcheval.metrics import BinaryAccuracy
from torchvision import tv_tensors
from torchvision.transforms import v2
from tqdm import tqdm

from data import attribute_indices
from eval.local_parts import attributes_indexes
from nets import PPConceptNet


In [ ]:
class CUBConceptDropDataset(Dataset):
    def __init__(self, data_root: str | Path, drop_attribute_index: int | None, crop_size: int = 50, check_integrity = False):
        self.data_root = Path(data_root)
        self.crop_size = crop_size
        self.drop_attribute_index = drop_attribute_index
        self.check_integrity = check_integrity
        with open(Path("data") / "CUB" / "class_attr_data_10" / "train.pkl", "rb") as fp:
            train_attribute_anns = pkl.load(fp)

        label2attr = dict()
        for ann in train_attribute_anns:
            label, attribute_vector = ann["class_label"], ann["attribute_label"]
            if label not in label2attr:
                label2attr[label] = attribute_vector

        self.attributes = torch.tensor([label2attr[i] for i in range(len(label2attr))], dtype=torch.long)

        with open(Path("data") / "CUB" / "cub_attributes_cleaned.txt", "r") as fp:
            all_attribute_texts = fp.read().splitlines()
        self.attribute_texts = [all_attribute_texts[i] for i in attribute_indices]

        images_df = pd.read_csv(
            Path(self.data_root) / "CUB_200_2011" / "images.txt",
            header=None,
            delimiter=" ",
            names=["img_id", "path"],
            usecols=[0, 1],
            index_col=0
        )
        splits_df = pd.read_csv(
            Path(self.data_root) / "CUB_200_2011" / "train_test_split.txt",
            header=None,
            delimiter=" ",
            names=["img_id", "is_train"],
            usecols=[0, 1],
            index_col=0
        )
        keypoints_df = pd.read_csv(
            Path(self.data_root) / "CUB_200_2011" / "parts"/ "part_locs.txt",
            header=None,
            delimiter=" ",
            names=["img_id", "part_idx", "x", "y"],
            usecols=[0, 1, 2, 3],
            index_col=0
        )
        bbox_df = pd.read_csv(
            Path(self.data_root) / "CUB_200_2011" / "bounding_boxes.txt",
            header=None,
            delimiter=" ",
            names=["img_id", "x", "y", "w", "h"],
            usecols=[0, 1, 2, 3, 4],
            index_col=0
        )
        images_df.index = images_df.index - 1
        splits_df.index = splits_df.index - 1
        keypoints_df.index = keypoints_df.index - 1
        keypoints_df["part_idx"] = keypoints_df["part_idx"] - 1
        bbox_df.index = bbox_df.index - 1

        self.images_df = images_df
        self.splits_df = splits_df
        self.keypoints_df = keypoints_df
        self.bbox_df = bbox_df

        self.samples_df = images_df.loc[splits_df["is_train"] == 0]

        # Process part name to idx mapping
        self.part_name2part_idx = defaultdict(list)
        with open(self.data_root / "CUB_200_2011" / "parts" / "parts.txt", "r") as fp:
            parts = [line.split(" ", 1)[1] for line in fp.read().splitlines()]
        for i, part_name in enumerate(parts):
            self.part_name2part_idx[part_name.split(" ")[-1]].append(i)

        with open(self.data_root / "CUB_200_2011" / "attributes.txt", "r") as fp:
            attributes = fp.read().splitlines()
            attributes = [attributes[i] for i in attributes_indexes]

        with open(self.data_root / "CUB_200_2011" / "parts" / "parts.txt", "r") as fp:
            parts = [line.split(" ", 1)[1] for line in fp.read().splitlines()]

        # Create a mapping from attribute index to part index
        self.attr_id2part_indices = defaultdict(list)
        for attr_idx, attr in enumerate(attributes):
            attr = attr.replace("bill", "beak")
            for part_name, part_indices in self.part_name2part_idx.items():
                if part_name in attr:
                    print("attr", f"{attr_idx}".ljust(3), " "*5, attr.split(" ")[1].ljust(40), "->", " " * 10, part_name)
                    self.attr_id2part_indices[attr_idx] += part_indices

        self.transforms = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize((0.485,0.456,0.406,),(0.229,0.224,0.225,),),
        ])

    def __len__(self):
        return len(self.samples_df)

    def __getitem__(self, index: int):
        im_id, im_path = self.samples_df.index[index], self.samples_df.iloc[index]["path"]
        label = int(im_path.split(".")[0]) - 1
        attr = self.attributes[label]

        im = Image.open(self.data_root / "cub200_cropped" / "test_cropped" / im_path)
        im = v2.functional.resize(im, [224, 224])

        if self.drop_attribute_index is None:
            return self.transforms(im), label, attr

        if self.check_integrity:
            assert attr[self.drop_attribute_index] == 1, f"Attribute to drop is not present as ground truth for sample {index}"

        sample_keypoints = self.keypoints_df.loc[
            (self.keypoints_df.index == im_id) &
            (self.keypoints_df["x"] != 0) &
            (self.keypoints_df["y"] != 0)
        ]
        raw_im = Image.open(self.data_root / "CUB_200_2011" / "images" / im_path)
        raw_w, raw_h = raw_im.size
        object_x, object_y, object_w, object_h = tuple(self.bbox_df.iloc[im_id][["x", "y", "w", "h"]])

        # Locate the part keypoints of the attribute, then apply crop and resize transforms on them
        part_indices = set(sample_keypoints['part_idx']) and set(self.attr_id2part_indices[self.drop_attribute_index])
        part_cxcy = sample_keypoints.loc[sample_keypoints["part_idx"].isin(part_indices)][["x", "y"]].to_numpy()
        part_cxcy_pt = tv_tensors.KeyPoints(part_cxcy, canvas_size=(raw_h, raw_w))
        part_cxcy_transformed = v2.functional.crop(part_cxcy_pt, top=object_y, left=object_x, height=object_h, width=object_w)
        part_cxcy_transformed = v2.functional.resize(part_cxcy_transformed, [224, 224])

        draw = ImageDraw.Draw(im)
        for kp in part_cxcy_transformed:
            draw.rectangle(xy=[(kp - self.crop_size // 2).tolist(), (kp + self.crop_size // 2).tolist()], fill="black")
        im_pt = self.transforms(im)

        return im_pt, label, attr

In [ ]:
num_classes, num_concepts = 200, 112
data_dir = Path("/Users/zhijiezhu/Developer/Research/datasets")

label2attr = dict()
with open(Path("data") / "CUB" / "class_attr_data_10" / "train.pkl", "rb") as fp:
    train_attribute_anns = pkl.load(fp)

for ann in train_attribute_anns:
    label, attribute_vector = ann["class_label"], ann["attribute_label"]
    if label not in label2attr:
        label2attr[label] = attribute_vector

attributes = torch.tensor([label2attr[i] for i in range(len(label2attr))], dtype=torch.long)

images_df = pd.read_csv(
    Path(data_dir) / "CUB_200_2011" / "images.txt",
    header=None,
    delimiter=" ",
    names=["img_id", "path"],
    usecols=[0, 1],
    index_col=0
)
splits_df = pd.read_csv(
    Path(data_dir) / "CUB_200_2011" / "train_test_split.txt",
    header=None,
    delimiter=" ",
    names=["img_id", "is_train"],
    usecols=[0, 1],
    index_col=0
)
images_df.index = images_df.index - 1
splits_df.index = splits_df.index - 1
images_df["label"] = images_df["path"].str.split(".").str[0].astype(int) - 1
samples_df = pd.concat([images_df, splits_df], axis=1)
samples_df = samples_df.loc[samples_df["is_train"] == 0]

dataset = CUBConceptDropDataset(
    data_root=Path(data_dir),
    drop_attribute_index=None
)
concept_drop_dataset = CUBConceptDropDataset(
    data_root=Path(data_dir),
    drop_attribute_index=0
)